In [ ]:
!pip uninstall -y mediapipe tensorflow keras protobuf tf-keras tensorflow-text ydf-tf

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tf_keras 2.20.0
Uninstalling tf_keras-2.20.0:
  Successfully uninstalled tf_keras-2.20.0
Found existing installation: tensorflow-text 2.20.1
Uninstalling tensorflow-text-2.20.1:
  Successfully uninstalled tensorflow-text-2.20.1
Found existing installation: ydf_tf 2.20.0
Uninstalling ydf_tf-2.20.0:
  Successfully uninstalled ydf_tf-2.20.0


In [ ]:
!pip install protobuf==4.25.3
!pip install mediapipe==0.10.14
!pip install tensorflow==2.19.1
!pip install keras==3.5.0
!pip install opencv-python scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-appengine-logging 1.9.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
googleapis-common-protos 1.75.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-bigtable 2.38.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.3 which is incompatible.
google-cloud-bigquery-storage 2.38.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
grpcio-sta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 842.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.5 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.20.0
    Uninstalling tensorboard-2.20.0:
      Successfully uninstalled tensorboard-2.20.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.20.0
    Uninstalling tensorflow-2.20.0:
      Successfully uninstalled tensorflow-2.20.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.1 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.1 which is incompatible.
tf-keras 2.20.0 requires tensorfl

In [ ]:
import os
import cv2
import pickle
import zipfile
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping
)

from tensorflow.keras.utils import (
    to_categorical
)

In [ ]:
import tensorflow as tf
import mediapipe as mp
import google.protobuf

print(
    tf.__version__
)

print(
    "MediaPipe Working"
)

print(
    google.protobuf.__version__
)

2.19.1
MediaPipe Working
4.25.3


In [ ]:
import os

print(
    os.listdir(
        "/content"
    )
)

['.config', 'model.keras', 'psllabels.json', '.ipynb_checkpoints', 'sample_data']


In [ ]:
import zipfile
import os

zip_path = "/content/psl_dataset_fingerspell.zip"

extract_path = "/content"

with zipfile.ZipFile(
    zip_path,
    "r"
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )

print("Dataset Extracted Successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/psl_dataset_fingerspell.zip'

In [ ]:
dataset_path = (
    "/content/"
    "new_psl_dataset"
)

print(
    os.listdir(
        dataset_path
    )[:10]
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/new_psl_dataset'

# **MediaPipe Hand Detection Setup**

In [ ]:
import mediapipe as mp

# Create MediaPipe Hands
mp_hands = mp.solutions.hands

# Hand detector
hands = mp_hands.Hands(

    static_image_mode=True,

    max_num_hands=1,

    min_detection_confidence=0.1
)

print(
    "MediaPipe Ready"
)

MediaPipe Ready


# Preproce img

In [ ]:
def preprocess_image(image):

    image = cv2.resize(
        image,
        (128,128)
    )

    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    return image_rgb

# **Normilization**

In [ ]:
def normalize_keypoints(kp):

    kp = kp.reshape(
        21,
        3
    )

    # wrist center
    kp = kp - kp[0]

    max_value = np.max(
        np.abs(kp)
    )

    if max_value > 0:

        kp = kp / max_value

    return kp.flatten()

# **Extract Keypoints**

In [ ]:
def extract_keypoints(image):

    image = preprocess_image(
        image
    )

    results = hands.process(
        image
    )

    if results.multi_hand_landmarks:

        landmarks = []

        hand = (
            results.multi_hand_landmarks[0]
        )

        for lm in hand.landmark:

            landmarks.extend([
                lm.x,
                lm.y,
                lm.z
            ])

        landmarks = np.array(
            landmarks
        )

        landmarks = normalize_keypoints(
            landmarks
        )

        return landmarks

    return None

# **Load Dataset + Extract Keypoints First**

In [ ]:
X = []
y = []

for label in sorted(
    os.listdir(dataset_path)
):

    folder_path = os.path.join(
        dataset_path,
        label
    )

    if not os.path.isdir(
        folder_path
    ):
        continue

    count = 0

    for image_name in os.listdir(
        folder_path
    ):

        image_path = os.path.join(
            folder_path,
            image_name
        )

        image = cv2.imread(
            image_path
        )

        if image is None:
            continue

        keypoints = extract_keypoints(
            image
        )

        if keypoints is not None:

            X.append(
                keypoints
            )

            y.append(
                label
            )

            count += 1

    print(
        f"{label}: {count} samples"
    )

print(
    "\nExtraction Complete"
)

print(
    "Total Samples:",
    len(X)
)

X = np.array(X)
y = np.array(y)

print(
    X.shape
)

print(
    y.shape
)

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


ain: 33 samples
alif: 57 samples
bari_yay: 59 samples
bay: 58 samples
chay: 59 samples
choti_yay: 57 samples
daal: 59 samples
ddaal: 38 samples
fay: 59 samples
gaff: 43 samples
ghain: 50 samples
hamza: 52 samples
hay: 57 samples
jeem: 59 samples
kaaf: 48 samples
khay: 59 samples
laam: 59 samples
meem: 55 samples
noon: 59 samples
pay: 55 samples
qaaf: 41 samples
ray: 38 samples
rray: 54 samples
say: 54 samples
seen: 53 samples
sheen: 34 samples
suaad: 59 samples
taay: 48 samples
toay: 55 samples
wow: 53 samples
zaal: 56 samples
zay: 46 samples
zoay: 34 samples
zuad: 33 samples
zzay: 38 samples

Extraction Complete
Total Samples: 1771
(1771, 63)
(1771,)


# **Keypoint Augmentation Function**

In [ ]:
def augment_keypoints(kp):

    points = kp.reshape(
        21,
        3
    ).copy()


    # 1. Random Noise


    noise = np.random.normal(
        0,
        0.03,
        points.shape
    )

    points += noise

    # 2. Random Scale

    scale = np.random.uniform(
        0.80,
        1.20
    )

    points *= scale


    # 3. Random Rotation


    angle = np.random.uniform(
        -20,
        20
    )

    rad = np.radians(
        angle
    )

    cos = np.cos(rad)
    sin = np.sin(rad)

    rotation = np.array([
        [cos, -sin],
        [sin, cos]
    ])

    points[:, :2] = (
        points[:, :2]
        @ rotation.T
    )

    # 4. Flip Hand


    if np.random.rand() > 0.5:

        points[:,0] = (
            -points[:,0]
        )


    # 5. Random Shift


    shift_x = np.random.uniform(
        -0.08,
        0.08
    )

    shift_y = np.random.uniform(
        -0.08,
        0.08
    )

    points[:,0] += shift_x
    points[:,1] += shift_y

    # 6. Small Landmark Variation

    random_move = np.random.normal(
        0,
        0.015,
        points.shape
    )

    points += random_move

    # Normalize Again

    max_value = np.max(
        np.abs(points)
    )

    if max_value > 0:

        points /= max_value

    return points.flatten()

# **TRAN TEST SPLIT**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_idx, y_test_idx = train_test_split(

    X,
    y_encoded,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded
)

print(
    X_train.shape
)

print(
    X_test.shape
)

(1416, 63)
(355, 63)


# **# `Augmented Dataset Creation`**

In [ ]:

X_aug = []
y_aug = []

for kp, label in zip(
    X_train,
    y_train_idx
):

    # Original
    X_aug.append(kp)
    y_aug.append(label)

    # 10 Extra Augmentations
    for i in range(10):

        aug = augment_keypoints(
            kp
        )

        X_aug.append(
            aug
        )

        y_aug.append(
            label
        )

X_train = np.array(
    X_aug
)

y_train_idx = np.array(
    y_aug
)

print(
    "Augmented Train Shape:",
    X_train.shape
)

Augmented Train Shape: (15576, 63)


In [ ]:
from collections import Counter

class_counts = Counter(
    y
)

print(
    "Before Filter:"
)

for label, count in class_counts.items():

    print(
        f"{label}: {count}"
    )

X_filtered = []
y_filtered = []

for x_item, y_item in zip(
    X,
    y
):

    if class_counts[
        y_item
    ] >= 2:

        X_filtered.append(
            x_item
        )

        y_filtered.append(
            y_item
        )

X = np.array(
    X_filtered
)

y = np.array(
    y_filtered
)

print(
    "\nAfter Filter:"
)

print(
    X.shape
)

print(
    y.shape
)

# **Label Encoder**

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(
    y
)

print(
    y_encoded.shape
)

(1771,)


# **BUILD MODEL**

In [ ]:
from tensorflow.keras.regularizers import l2

model = Sequential([

    Dense(
        512,
        activation='relu',
        input_shape=(63,),
        kernel_regularizer=l2(0.001)
    ),

    BatchNormalization(),

    Dropout(0.5),

    Dense(
        256,
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),

    BatchNormalization(),

    Dropout(0.4),

    Dense(
        128,
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),

    BatchNormalization(),

    Dropout(0.3),

    Dense(
        64,
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),

    BatchNormalization(),

    Dropout(0.2),

    Dense(
        len(np.unique(y)),
        activation='softmax'
    )
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 35)             │         2,275 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 211,363 (825.64 KB)

 Trainable params: 209,443 (818.14 KB)

 Non-trainable params: 1,920 (7.50 KB)

# **Compile**

In [ ]:
model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

# **TRAIN MODEL**

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(
    y_train_idx
)

history = model.fit(

    X_train,
    y_train_cat,

    validation_split=0.2,

    epochs=70,

    batch_size=16,

    callbacks=[early_stop]
)

Epoch 1/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.1772 - loss: 3.7588 - val_accuracy: 0.5761 - val_loss: 2.1064
Epoch 2/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.4679 - loss: 2.3185 - val_accuracy: 0.7221 - val_loss: 1.6119
Epoch 3/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5480 - loss: 1.9817 - val_accuracy: 0.7282 - val_loss: 1.4427
Epoch 4/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.6036 - loss: 1.7602 - val_accuracy: 0.7372 - val_loss: 1.4681
Epoch 5/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.6411 - loss: 1.6222 - val_accuracy: 0.7776 - val_loss: 1.2260
Epoch 6/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.6747 - loss: 1.4960 - val_accuracy: 0.8171 - val_loss: 1.1471
Epoch 7/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.6934 - loss: 1.4151 - val_accuracy: 0.8058 - val_loss: 1.1272
Epoch 8/70
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.6994 - loss: 1.3862 - val_accuracy: 

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(
    "Test Accuracy:",
    accuracy * 100
)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9336 - loss: 0.7763 
Test Accuracy: 93.80281567573547


In [ ]:
model.save(
    "psl_model.keras"
)

print("Model Saved Successfully")

Model Saved Successfully


# **Test**

In [ ]:
import json

with open(
    "psllabels.json",
    "w"
) as f:

    json.dump(
        {str(i): label for i, label in enumerate(encoder.classes_)},
        f,
        indent=4
    )

print(
    "Labels Saved Successfully"
)

NameError: name 'encoder' is not defined

In [ ]:
import cv2
import json
import mediapipe as mp
import numpy as np

from google.colab import files
from tensorflow.keras.models import load_model

# =====================================
# LOAD MODEL
# =====================================

model = load_model(
    "/content/psl_model Final 93%+.keras"
)

print("Model Loaded")

# =====================================
# LOAD LABELS JSON
# =====================================

with open(
    "/content/psllabels.json",
    "r"
) as f:

    labels = json.load(f)

print("Labels Loaded")

# =====================================
# UPLOAD IMAGE
# =====================================

uploaded = files.upload()

image_path = list(
    uploaded.keys()
)[0]

# =====================================
# READ IMAGE
# =====================================

image = cv2.imread(
    image_path
)

if image is None:

    raise Exception(
        "Image Not Found"
    )

# =====================================
# PREPROCESS IMAGE
# =====================================

image = cv2.resize(
    image,
    (256,256)
)

image = cv2.convertScaleAbs(
    image,
    alpha=1.3,
    beta=20
)

image = cv2.GaussianBlur(
    image,
    (3,3),
    0
)

rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

# =====================================
# MEDIAPIPE
# =====================================

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(

    static_image_mode=True,

    max_num_hands=1,

    min_detection_confidence=0.3
)

# =====================================
# NORMALIZATION
# =====================================

def normalize_keypoints(kp):

    kp = kp.reshape(
        21,
        3
    )

    kp = kp - kp[0]

    max_value = np.max(
        np.abs(kp)
    )

    if max_value > 0:

        kp = kp / max_value

    return kp.flatten()

# =====================================
# DETECT HAND
# =====================================

results = hands.process(
    rgb
)

# =====================================
# PREDICTION
# =====================================

if results.multi_hand_landmarks:

    landmarks = []

    hand = (
        results.multi_hand_landmarks[0]
    )

    for lm in hand.landmark:

        landmarks.extend([
            lm.x,
            lm.y,
            lm.z
        ])

    landmarks = np.array(
        landmarks
    )

    landmarks = normalize_keypoints(
        landmarks
    )

    X_pred = landmarks.reshape(
        1,
        -1
    )

    prediction = model.predict(
        X_pred,
        verbose=0
    )[0]

    confidence = (
        np.max(prediction)
        * 100
    )

    predicted_index = np.argmax(
        prediction
    )

    # JSON labels
    try:

        predicted_label = labels[
            str(predicted_index)
        ]

    except:

        predicted_label = labels[
            predicted_index
        ]

    if confidence < 70:

        print(
            "\nUnknown Sign"
        )

        print(
            f"Confidence: {confidence:.2f}%"
        )

    else:

        print(
            "\nPrediction:",
            predicted_label
        )

        print(
            f"Confidence: {confidence:.2f}%"
        )

else:

    print(
        "No Hand Detected"
    )

Model Loaded
Labels Loaded


Saving WIN_20260530_00_01_44_Pro.jpg to WIN_20260530_00_01_44_Pro (1).jpg



Prediction: bay
Confidence: 85.03%


Model Loaded Successfully


## **LIVE TEST**

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from google.colab.patches import cv2_imshow

import cv2
import numpy as np
import json
import mediapipe as mp

from tensorflow.keras.models import load_model

# =====================================
# LOAD MODEL
model = load_model(
    "/content/psl_model Final 93%+.keras"
)

# =====================================
# LOAD LABELS
# =====================================

with open(
    "/content/psllabels.json",
    "r"
) as f:

    labels = json.load(f)

print("Model Loaded")
print("Labels Loaded")

# =====================================
# MEDIAPIPE
# =====================================

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(

    static_image_mode=True,

    max_num_hands=1,

    min_detection_confidence=0.3
)

# =====================================
# NORMALIZATION
# =====================================

def normalize_keypoints(kp):

    kp = kp.reshape(
        21,
        3
    )

    kp = kp - kp[0]

    max_value = np.max(
        np.abs(kp)
    )

    if max_value > 0:

        kp = kp / max_value

    return kp.flatten()

# =====================================
# CAMERA
# =====================================

def take_photo():

    js = Javascript('''

    async function takePhoto() {

      const div = document.createElement('div');

      const capture =
        document.createElement('button');

      capture.textContent = 'Capture';

      div.appendChild(capture);

      const video =
        document.createElement('video');

      video.style.display = 'block';

      const stream =
        await navigator.mediaDevices.getUserMedia(
          {video:true}
        );

      document.body.appendChild(div);

      div.appendChild(video);

      video.srcObject = stream;

      await video.play();

      google.colab.output.setIframeHeight(
        document.body.scrollHeight,
        true
      );

      await new Promise(
        (resolve)=>
        capture.onclick=resolve
      );

      const canvas =
        document.createElement('canvas');

      canvas.width =
        video.videoWidth;

      canvas.height =
        video.videoHeight;

      canvas.getContext('2d')
      .drawImage(video,0,0);

      stream.getTracks()
      .forEach(
        track=>track.stop()
      );

      div.remove();

      return canvas.toDataURL(
        'image/jpeg'
      );
    }

    ''')

    display(js)

    data = eval_js(
        'takePhoto()'
    )

    binary = b64decode(
        data.split(',')[1]
    )

    image_np = np.frombuffer(
        binary,
        np.uint8
    )

    image = cv2.imdecode(
        image_np,
        cv2.IMREAD_COLOR
    )

    return image

# =====================================
# TAKE PHOTO
# =====================================

image = take_photo()

rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

results = hands.process(
    rgb
)

prediction_text = "No Hand"

# =====================================
# PREDICTION
# =====================================

if results.multi_hand_landmarks:

    hand = results.multi_hand_landmarks[0]

    landmarks = []

    mp_draw.draw_landmarks(

        image,

        hand,

        mp_hands.HAND_CONNECTIONS,

        mp_draw.DrawingSpec(
            color=(0,255,0),
            thickness=2
        ),

        mp_draw.DrawingSpec(
            color=(255,0,0),
            thickness=2,
            circle_radius=5
        )
    )

    h, w, _ = image.shape

    for lm in hand.landmark:

        x = int(lm.x * w)
        y = int(lm.y * h)

        cv2.circle(
            image,
            (x,y),
            7,
            (0,0,255),
            -1
        )

        landmarks.extend([
            lm.x,
            lm.y,
            lm.z
        ])

    landmarks = np.array(
        landmarks
    )

    landmarks = normalize_keypoints(
        landmarks
    )

    X_pred = landmarks.reshape(
        1,
        -1
    )

    prediction = model.predict(
        X_pred,
        verbose=0
    )[0]

    confidence = (
        np.max(prediction)
        * 100
    )

    predicted_index = np.argmax(
        prediction
    )

    predicted_label = labels[
        str(predicted_index)
    ]

    if confidence > 70:

        prediction_text = (
            f"{predicted_label}"
            f" ({confidence:.2f}%)"
        )

    else:

        prediction_text = (
            "Unknown Sign"
        )

# =====================================
# SHOW RESULT
# =====================================

cv2.putText(

    image,

    prediction_text,

    (20,50),

    cv2.FONT_HERSHEY_SIMPLEX,

    1,

    (0,255,0),

    2
)

cv2_imshow(
    image
)

print(
    "Prediction:",
    prediction_text
)

Model Loaded
Labels Loaded


<IPython.core.display.Javascript object>

In [ ]:
import os

print(
    os.path.exists("/content/psl.keras")
)

True


# ***FINE TUNE MODEL***

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/psl_model.keras"
)

print("Old Model Loaded")

Old Model Loaded


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(

    optimizer=Adam(
        learning_rate=0.00005
    ),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(
    y_train_idx
)

history = model.fit(

    X_train,
    y_train_cat,

    validation_split=0.2,

    epochs=25,

    batch_size=16,

    callbacks=[early_stop]
)

Epoch 1/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8840 - loss: 0.5961 - val_accuracy: 0.9239 - val_loss: 0.5730
Epoch 2/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8803 - loss: 0.5935 - val_accuracy: 0.9249 - val_loss: 0.5761
Epoch 3/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8851 - loss: 0.5830 - val_accuracy: 0.9236 - val_loss: 0.5764
Epoch 4/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8778 - loss: 0.5897 - val_accuracy: 0.9272 - val_loss: 0.5713
Epoch 5/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8844 - loss: 0.5959 - val_accuracy: 0.9284 - val_loss: 0.5638
Epoch 6/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8828 - loss: 0.5809 - val_accuracy: 0.9275 - val_loss: 0.5745
Epoch 7/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8756 - loss: 0.5880 - val_accuracy: 0.9255 - val_loss: 0.5626
Epoch 8/25
779/779 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8926 - loss: 0.5500 - val_accuracy: 

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(
    "Test Accuracy:",
    accuracy * 100
)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9536 - loss: 0.4388 
Test Accuracy: 95.21126747131348


In [ ]:
model.save(
    "psl_model_finetuned.keras"
)

print("Fine Tuned Model Saved")

Fine Tuned Model Saved


In [ ]:
import cv2
import pickle
import mediapipe as mp
import numpy as np
from google.colab import files
from tensorflow.keras.models import load_model

# =====================================
# Load Model
# =====================================

model = load_model(
    "/content/asl_model.keras"
)

print("Model Loaded")

# =====================================
# Load Label Encoder
# =====================================

with open(
    "/content/label_encoder.pkl",
    "rb"
) as f:

    encoder = pickle.load(f)

print("Label Encoder Loaded")

# =====================================
# Upload Image
# =====================================

uploaded = files.upload()

image_path = list(
    uploaded.keys()
)[0]

# =====================================
# Read Image
# =====================================

image = cv2.imread(
    image_path
)

# =====================================
# Better Preprocessing
# =====================================

image = cv2.resize(
    image,
    (256,256)
)

# Brightness Improve
image = cv2.convertScaleAbs(
    image,
    alpha=1.3,
    beta=20
)

# Gaussian Blur
image = cv2.GaussianBlur(
    image,
    (3,3),
    0
)

# RGB Convert
rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

# =====================================
# MediaPipe Setup
# =====================================

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3
)

results = hands.process(rgb)

# =====================================
# Prediction
# =====================================

if results.multi_hand_landmarks:

    landmarks = []

    for hand in results.multi_hand_landmarks:

        for lm in hand.landmark:

            landmarks.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    X = np.array(
        landmarks
    ).reshape(1,-1)

    # Normalize
    X = (
        X - np.mean(X)
    ) / np.std(X)

    prediction = model.predict(
        X,
        verbose=0
    )

    confidence = np.max(
        prediction
    ) * 100

    predicted_index = np.argmax(
        prediction
    )

    predicted_label = encoder.inverse_transform(
        [predicted_index]
    )[0]

    if confidence < 75:

        print(
            "Unknown Sign"
        )

    else:

        print(
            "Prediction:",
            predicted_label
        )

        print(
            f"Confidence: {confidence:.2f}%"
        )

else:

    print(
        "No Hand Detected"
    )